## 1. Mask R-CNN

**Contents:**
1. What & Why
2. Architecture
3. The Mask Branch & RoIAlign
4. Loss
5. torchvision Usage
6. References

---

**Mask R-CNN** (He, Gkioxari, Dollar, Girshick, ICCV 2017) extends **Faster R-CNN** from object detection to **instance segmentation**: it not only draws a bounding box and label for each object, but also predicts a **per-pixel binary mask** outlining the exact shape of every individual object instance.

The key insight is simple and powerful: keep the entire Faster R-CNN detector and add a **third, parallel output branch** that predicts a mask for each RoI. The two contributions that make this work well are:

1. adding a small **fully-convolutional mask head** alongside the existing box/class heads, and
2. replacing RoIPool with **RoIAlign**, whose precise feature sampling is essential for pixel-accurate masks.

**Instance** segmentation (Mask R-CNN) separates each object as its own entity — two overlapping people get two distinct masks — unlike **semantic** segmentation, which only labels each pixel by class and cannot tell instances apart.

## 2. Architecture

```
Image -> Backbone + FPN -> RPN -> proposals
                                    |
                                    v
                                RoIAlign
                                    |
            +-----------------------+------------------------+
            v                       v                        v
       Box Head               Class Head                Mask Head
   (refined box)           (object class)         (small FCN -> 28x28 mask
                                                    per class)
```

Everything up to and including RoIAlign is **identical to Faster R-CNN** (backbone, FPN, RPN, two-stage design). The new piece is the **mask head**: a small **fully convolutional network (FCN)** applied to each RoI's aligned feature. It outputs `C` masks of resolution `m x m` (commonly 28x28), one mask per class. At inference, only the mask for the predicted class is used, then it is resized back to the box.

Decoupling masks per class (rather than one mask competing across classes) avoids inter-class competition and was shown to improve quality.

## 3. The Mask Branch & RoIAlign

**Why RoIAlign matters here.** RoIPool rounds (quantizes) RoI coordinates to the feature grid twice. For box detection a few pixels of misalignment barely matter, but for **per-pixel masks** it is harmful. **RoIAlign** removes all quantization: for each of the `m x m` output bins it samples feature values at exact continuous locations using **bilinear interpolation**. This pixel-to-pixel alignment is the single biggest reason Mask R-CNN produces sharp masks, and it also improves the box AP.

**Mask head output.** For each RoI the head produces a `C x m x m` tensor. Each of the `m x m` values passes through a **per-pixel sigmoid** (not a softmax over classes), giving an independent foreground probability for every pixel of every class. The mask is decoupled from classification: the class head decides *what* the object is; the mask head only has to decide *which pixels* belong to it.

## 4. Loss

Mask R-CNN's loss adds a mask term to the Faster R-CNN multi-task loss:

```
L = L_cls + L_box + L_mask
```

- **L_cls, L_box** are exactly as in Faster R-CNN (cross-entropy over classes; smooth L1 on box deltas).
- **L_mask** is the **average binary cross-entropy** over the `m x m` pixels — but computed **only on the mask of the ground-truth class** `k`. Masks of other classes do not contribute to the loss.

This per-class, per-pixel sigmoid formulation (rather than a softmax over classes per pixel) is what lets each class's mask be predicted independently and avoids competition between classes.

## 5. torchvision Usage

```python
import torch
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn,
    MaskRCNN_ResNet50_FPN_Weights,
)

weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
model = maskrcnn_resnet50_fpn(weights=weights)
model.eval()
preprocess = weights.transforms()

from torchvision.io import read_image
img = read_image("image.jpg")            # uint8 CxHxW
batch = [preprocess(img)]

with torch.no_grad():
    out = model(batch)[0]

print(out["boxes"])   # (N, 4)  [x1, y1, x2, y2]
print(out["labels"])  # (N,)   COCO class ids
print(out["scores"])  # (N,)   confidences
print(out["masks"])   # (N, 1, H, W) soft masks in [0, 1]; threshold ~0.5 for binary

# Binarize the masks of confident detections
keep = out["scores"] > 0.5
binary_masks = out["masks"][keep] > 0.5
```

**Fine-tuning** requires targets that include `masks` (a `(N, H, W)` uint8 tensor of per-instance binary masks) in addition to `boxes` and `labels`. To change the number of classes, swap both the box predictor and the mask predictor:

```python
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

num_classes = 2  # e.g. background + 1 object

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
hidden_layer = 256
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)
```

An improved recipe is available as `maskrcnn_resnet50_fpn_v2`.

## 6. References

- He, Gkioxari, Dollar, Girshick, *Mask R-CNN*, ICCV 2017 — https://arxiv.org/abs/1703.06870
- Ren et al., *Faster R-CNN*, NeurIPS 2015 — https://arxiv.org/abs/1506.01497
- torchvision detection models — https://pytorch.org/vision/stable/models.html#instance-segmentation
- torchvision finetuning (Mask R-CNN) tutorial — https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html